# 08. Normalization Techniques Practice

这一练习覆盖 BatchNorm、LayerNorm 和运行统计量更新。

In [ ]:
import torch
import torch.nn.functional as F


In [ ]:
def batch_norm_train(x, gamma, beta, eps=1e-5):
    batch_mean = x.mean(dim=0)
    batch_var = x.var(dim=0, unbiased=False)
    normalized = (x - batch_mean) / torch.sqrt(batch_var + eps)
    return normalized * gamma + beta, batch_mean, batch_var


def batch_norm_eval(x, gamma, beta, running_mean, running_var, eps=1e-5):
    normalized = (x - running_mean) / torch.sqrt(running_var + eps)
    return normalized * gamma + beta


def layer_norm_last_dim(x, gamma, beta, eps=1e-5):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, unbiased=False, keepdim=True)
    normalized = (x - mean) / torch.sqrt(var + eps)
    return normalized * gamma + beta


def update_running_stats(running_mean, running_var, batch_mean, batch_var, momentum=0.1):
    new_mean = (1.0 - momentum) * running_mean + momentum * batch_mean
    new_var = (1.0 - momentum) * running_var + momentum * batch_var
    return new_mean, new_var


In [ ]:
def test_batch_norm_train():
    x = torch.tensor([[1.0, 2.0, 3.0], [2.0, 4.0, 6.0]])
    gamma = torch.tensor([1.0, 1.5, 2.0])
    beta = torch.tensor([0.0, 1.0, -1.0])
    y, mean, var = batch_norm_train(x, gamma, beta)
    expected_mean = torch.tensor([1.5, 3.0, 4.5])
    expected_var = torch.tensor([0.25, 1.0, 2.25])
    expected = (x - expected_mean) / torch.sqrt(expected_var + 1e-5) * gamma + beta
    assert torch.allclose(mean, expected_mean)
    assert torch.allclose(var, expected_var)
    assert torch.allclose(y, expected, atol=1e-6, rtol=1e-6)
    print('✅ batch_norm_train 通过')


def test_batch_norm_eval():
    x = torch.tensor([[1.0, 2.0, 3.0]])
    gamma = torch.tensor([1.0, 1.0, 1.0])
    beta = torch.tensor([0.0, 0.0, 0.0])
    running_mean = torch.tensor([1.0, 2.0, 3.0])
    running_var = torch.tensor([1.0, 4.0, 9.0])
    y = batch_norm_eval(x, gamma, beta, running_mean, running_var)
    assert torch.allclose(y, torch.zeros_like(x), atol=1e-6, rtol=1e-6)
    print('✅ batch_norm_eval 通过')


def test_layer_norm_last_dim():
    x = torch.tensor([[1.0, 2.0, 3.0], [2.0, 2.0, 2.0]])
    gamma = torch.tensor([1.0, 2.0, 3.0])
    beta = torch.tensor([0.5, 0.0, -0.5])
    y = layer_norm_last_dim(x, gamma, beta)
    expected = F.layer_norm(x, x.shape[-1:], gamma, beta, eps=1e-5)
    assert torch.allclose(y, expected, atol=1e-6, rtol=1e-6)
    print('✅ layer_norm_last_dim 通过')


def test_update_running_stats():
    running_mean = torch.tensor([0.0, 0.0, 0.0])
    running_var = torch.tensor([1.0, 1.0, 1.0])
    batch_mean = torch.tensor([2.0, 4.0, 6.0])
    batch_var = torch.tensor([3.0, 5.0, 7.0])
    new_mean, new_var = update_running_stats(running_mean, running_var, batch_mean, batch_var, momentum=0.5)
    assert torch.allclose(new_mean, torch.tensor([1.0, 2.0, 3.0]))
    assert torch.allclose(new_var, torch.tensor([2.0, 3.0, 4.0]))
    print('✅ update_running_stats 通过')


test_batch_norm_train()
test_batch_norm_eval()
test_layer_norm_last_dim()
test_update_running_stats()


In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0], [3.0, 4.0, 5.0]])
gamma = torch.ones(3)
beta = torch.zeros(3)
y, mean, var = batch_norm_train(x, gamma, beta)
print('BatchNorm 输出：')
print(y)
print('batch mean:', mean.tolist())
print('batch var:', var.tolist())
print()
print('LayerNorm 输出：')
print(layer_norm_last_dim(x, gamma, beta))
